# Vista batch session — semantic-geometric-decoupling-framework

Plug-and-play driver for **one interactive Jupyter session** (~2h walltime slot) inside a
larger Vista allocation window (e.g. 22h). Open it in the GH200 container kernel and
**Run All** — the setup cells self-heal the environment before anything else runs:

- strip stray `PYTHONPATH`/`sys.path` entries left by other Python versions (and clean
  the environment the runner subprocess will inherit)
- `cd` to the repo root no matter which directory the notebook was opened from
- default `HF_HOME`/`HF_HUB_OFFLINE` to the prefetched `$SCRATCH/hf` cache
- probe every runtime dependency in a subprocess and auto-install anything
  missing/broken into this kernel's user site (`numpy<2` and headless-OpenCV pins
  included; `peft`/`accelerate`/`transformers` are era-matched to the container's
  torch and installed so pip can never replace the CUDA build)
- stage user-space X11/OpenGL libraries when the node lacks them — `open3d` links
  `libX11`/`libGL` even for headless raycasting, and there is no root to install them
- rebuild the experiment queue if its statuses were committed on another machine, and
  idempotently merge the H1×H3×H5 hypothesis grid
- verify CUDA with a **real matmul** — device visibility alone reports PASS even on a
  build with no kernels for this GPU

Remaining one-time prerequisites (login node, internet egress): clone the repo under
`$WORK`, then `bash scripts/prefetch_models.sh` to stage the container image and Qwen
weights.

**Per session:** Run All → checklists gate the canary → runner drains the queue under a
walltime-matched `SIGTERM` timeout → progress is verified against fresh checkpoints → a
handoff entry records state for the next session. When the session ends, relaunch the
Jupyter app and Run All again — progress carries over via the queue + checkpoints.


## Setup A — interpreter & path sanitization

Stdlib-only; must run before any third-party import so a polluted `PYTHONPATH` can't load ABI-incompatible packages into the kernel.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

_PYVER = f"python{sys.version_info.major}.{sys.version_info.minor}"


def _foreign(entry: str) -> bool:
    return bool(re.search(r"python3\.\d+", entry)) and _PYVER not in entry


# A PYTHONPATH aimed at another interpreter's site-packages (seen on Vista: a
# python3.11 user dir leaking onto this 3.10 kernel) shadows numpy/cv2/... with
# ABI-incompatible builds. Repair sys.path (this kernel) and os.environ (inherited
# by pip and by the runner subprocess in Checklist 4).
_stripped = [p for p in sys.path if _foreign(p)]
for _p in _stripped:
    sys.path.remove(_p)
_pp = [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
_pp_kept = [p for p in _pp if not _foreign(p)]
if len(_pp_kept) != len(_pp):
    if _pp_kept:
        os.environ["PYTHONPATH"] = os.pathsep.join(_pp_kept)
    else:
        os.environ.pop("PYTHONPATH", None)


# Jupyter starts kernels in the notebook's own directory — walk up to the repo root.
def _find_repo_root(start: Path):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return None


_root = _find_repo_root(Path.cwd())
if _root is not None and _root != Path.cwd():
    os.chdir(_root)
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Offline HuggingFace defaults: weights live in the $SCRATCH/hf cache staged by
# scripts/prefetch_models.sh; compute nodes must never touch the network.
if os.environ.get("SCRATCH"):
    os.environ.setdefault("HF_HOME", os.path.join(os.environ["SCRATCH"], "hf"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")

checklist = []  # (label, ok, detail) accumulated across the whole session


def check(label, ok, detail=""):
    checklist.append((label, ok, detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f" — {detail}" if detail else ""))
    return ok


check(
    "interpreter paths sane",
    True,
    f"auto-removed {_stripped} — fix the PYTHONPATH line in ~/.bashrc to silence this"
    if _stripped
    else "clean",
)
check("cwd is repo root", (REPO_ROOT / "pyproject.toml").is_file(), str(REPO_ROOT))
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError(
        "could not find pyproject.toml here or in any parent — open this notebook "
        "from inside the cloned repo"
    )
print(f"python: {sys.executable} ({_PYVER})")
print(
    f"HF_HOME={os.environ.get('HF_HOME', '<unset>')}  HF_HUB_OFFLINE={os.environ['HF_HUB_OFFLINE']}"
)

## Setup B — dependency probe & auto-install

Probes every runtime dependency in a **subprocess** (so a broken package can't taint this kernel), auto-installs what's missing into the kernel's own user site (persists in `$HOME` — later sessions skip straight through), then re-probes. `peft`/`accelerate` are installed `--no-deps` because they hard-depend on torch: pip must never replace the container's CUDA build with a PyPI wheel.

In [ ]:
import platform

_REQUIRED_LIBS = [
    "numpy",
    "cv2",
    "scipy.ndimage",
    "yaml",
    "PIL.Image",
    "open3d",
    "webdataset",
    "transformers",
    "peft",
    "accelerate",
    "qwen_vl_utils",
]
_OPTIONAL_LIBS = ["plotly", "sklearn", "langgraph", "datasets"]
_PROJECT_MODS = [
    "engine_grounder",
    "training.runner",
    "training.datagen",
    "training.train_pointnet",
    "training.train_lora",
    "training.eval",
]

# open3d's wheels link libX11/libGL even for pure headless raycasting; compute nodes
# have neither and there is no root to apt-install them. stage_headless_gl_libs.py
# downloads the Ubuntu runtime libs into $HOME; here we make them resolvable for
# this kernel (ctypes preload — the loader reuses already-loaded sonames when
# open3d's extension is dlopened later) and for subprocesses (LD_LIBRARY_PATH,
# which glibc reads at process startup).
_SYSLIB_DEST = Path.home() / ".local" / "headless-gl-libs"
_SYSLIB_DIR = _SYSLIB_DEST / "usr" / "lib" / f"{platform.machine()}-linux-gnu"
_GL_PRELOAD_ORDER = [
    "libXau.so.6",
    "libXdmcp.so.6",
    "libxcb.so.1",
    "libX11.so.6",
    "libXext.so.6",
    "libGLdispatch.so.0",
    "libGLX.so.0",
    "libGL.so.1",
    "libOpenGL.so.0",
    "libEGL.so.1",
]


def _activate_headless_gl():
    import ctypes

    if _SYSLIB_DIR.is_dir():
        _prev = os.environ.get("LD_LIBRARY_PATH", "")
        if str(_SYSLIB_DIR) not in _prev.split(os.pathsep):
            os.environ["LD_LIBRARY_PATH"] = str(_SYSLIB_DIR) + (os.pathsep + _prev if _prev else "")
    missing = []
    for _name in _GL_PRELOAD_ORDER:
        try:
            ctypes.CDLL(_name, mode=ctypes.RTLD_GLOBAL)
            continue
        except OSError:
            pass
        _staged = _SYSLIB_DIR / _name
        try:
            if _staged.exists():
                ctypes.CDLL(str(_staged), mode=ctypes.RTLD_GLOBAL)
                continue
        except OSError:
            pass
        missing.append(_name)
    return missing


def _system_x11_loads():
    import ctypes

    try:
        ctypes.CDLL("libX11.so.6")
        return True
    except OSError:
        return False


if _SYSLIB_DIR.is_dir():
    _activate_headless_gl()

_PROBE = (
    (
        "import importlib, json, sys\n"
        f"required = {_REQUIRED_LIBS!r}\n"
        f"optional = {_OPTIONAL_LIBS!r}\n"
        f"project = {_PROJECT_MODS!r}\n"
    )
    + """
out = {}


def _try(name):
    try:
        importlib.import_module(name)
        top = sys.modules[name.split(".")[0]]
        return ["ok", str(getattr(top, "__version__", ""))]
    except Exception as exc:
        return ["fail", type(exc).__name__ + ": " + str(exc)]


for _n in required + optional + project:
    out[_n] = _try(_n)
if out["numpy"][0] == "ok":
    import numpy

    if tuple(int(x) for x in numpy.__version__.split(".")[:2]) >= (2, 0):
        out["numpy"] = [
            "fail",
            numpy.__version__ + " (need <2: container scipy/torch are numpy1-ABI)",
        ]
hard_ok = all(out[_n][0] == "ok" for _n in required + project)
print("PROBE_JSON:" + json.dumps(out))
sys.exit(0 if hard_ok else 1)
"""
)


def _run_probe():
    proc = subprocess.run(
        [sys.executable, "-c", _PROBE], capture_output=True, text=True, cwd=str(REPO_ROOT)
    )
    results = {}
    for line in proc.stdout.splitlines():
        if line.startswith("PROBE_JSON:"):
            results = json.loads(line[len("PROBE_JSON:") :])
    return proc.returncode == 0 and bool(results), results


def _report(results):
    for name in _REQUIRED_LIBS + _PROJECT_MODS:
        status, detail = results.get(name, ("fail", "no probe result (probe crashed?)"))
        print(f"  {'ok  ' if status == 'ok' else 'FAIL'} {name} {detail}")
    for name in _OPTIONAL_LIBS:
        status, detail = results.get(name, ("fail", "no probe result"))
        print(f"  {'ok  ' if status == 'ok' else 'warn'} {name} {detail} (optional)")


deps_ok, probe_results = _run_probe()
if not deps_ok:
    print("dependency probe failed — auto-repairing (pip user site + headless GL libs):")
    _report(probe_results)
    _CORE = [
        "numpy>=1.24,<2",
        "opencv-python-headless>=4.7,<5",
        "open3d>=0.17",
        "pyyaml>=6.0",
        "Pillow>=9.5",
        "webdataset>=0.2",
        "transformers>=4.45,<4.50",
        "qwen-vl-utils>=0.0.8",
        "psutil",
        "plotly>=5.14",
        "scikit-learn>=1.3",
        "langgraph>=0.2",
        "datasets>=2.19",
    ]
    # peft/accelerate hard-depend on torch: --no-deps plus era-matched pins, so pip
    # can neither replace the container's CUDA torch nor pick releases that import
    # symbols it lacks (peft>=0.15 wants torch.distributed.tensor.DTensor, absent
    # from the NGC 24.10 torch 2.5.0a0 snapshot — transformers is pinned <4.50
    # above for the same era-matching reason).
    _TORCH_DEPENDENT = ["peft>=0.13,<0.15", "accelerate>=1.0,<1.3"]
    for args in (
        ["uninstall", "-y", "opencv-python"],  # GUI build needs libxcb; headless replaces it
        ["install", "--user", "-q", *_CORE],
        ["install", "--user", "-q", "--no-deps", *_TORCH_DEPENDENT],
    ):
        proc = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
        print(f"$ pip {' '.join(args[:2])} ... (exit {proc.returncode})")
        for line in (proc.stdout + proc.stderr).strip().splitlines()[-8:]:
            print("   ", line)
    if not _system_x11_loads():
        print("staging headless X11/OpenGL runtime libraries (no root required):")
        _stage = subprocess.run(
            [
                sys.executable,
                str(REPO_ROOT / "scripts" / "stage_headless_gl_libs.py"),
                "--dest",
                str(_SYSLIB_DEST),
            ],
            capture_output=True,
            text=True,
        )
        for line in (_stage.stdout + _stage.stderr).strip().splitlines()[-14:]:
            print("   ", line)
    _unresolved = _activate_headless_gl()
    if _unresolved:
        print(f"warning: still could not preload {_unresolved}")
    deps_ok, probe_results = _run_probe()

_report(probe_results)
check("all required deps + project modules import cleanly", deps_ok)
if not deps_ok:
    raise RuntimeError(
        "dependencies still broken after auto-repair. If this node lacks internet "
        "egress, run once from a login node:\n"
        "  apptainer exec $SCRATCH/containers/pytorch-arm64.sif python -m pip install "
        "--user 'numpy>=1.24,<2' 'opencv-python-headless>=4.7,<5' 'open3d>=0.17' "
        "pyyaml Pillow webdataset 'transformers>=4.45,<4.50' qwen-vl-utils psutil\n"
        "  apptainer exec $SCRATCH/containers/pytorch-arm64.sif python -m pip install "
        "--user --no-deps 'peft>=0.13,<0.15' 'accelerate>=1.0,<1.3'\n"
        "  apptainer exec $SCRATCH/containers/pytorch-arm64.sif python "
        "scripts/stage_headless_gl_libs.py"
    )

## Config — review before the first session of the window

In [ ]:
PROFILE_NAME = "vista"
QUEUE_PATH = "experiments/queue.yaml"

# Wall-clock budget for *this* session. Keep a safety buffer below the app's actual
# walltime request so SIGTERM (checkpoint-and-stop) fires before OnDemand SIGKILLs the
# kernel outright — the runner traps SIGTERM and checkpoints; it cannot trap SIGKILL.
SESSION_MINUTES = 110
SAFETY_BUFFER_MINUTES = 10
KILL_AFTER_SECONDS = 90  # grace period between SIGTERM and a forced SIGKILL, as in local_chain.sh

# Total allocation window this session is one slice of. Only used for the human-readable
# budget readout below — it does not talk to the TACC allocation system.
WINDOW_HOURS = 22.0

# Off by default: the static H1 x H3 x H5 grid runs first, unattended-loop planner
# refill second, once the static grid has been proven end-to-end (see
# training/runner.py:_default_evolving_scheduler and slurm/chain_job.sh).
EVOLVE = False

# Minimum acceptable canary IoU and GPU utilization before we trust a session made
# real progress (not just "a process ran").
MIN_CANARY_IOU = 0.30
LOW_GPU_UTIL_WARN_PCT = 20.0

## Checklist 1 — Environment

In [ ]:
for var in ("SCRATCH", "WORK"):
    check(
        f"${var} is set",
        var in os.environ,
        os.environ.get(var, "<unset — vista profile paths fall back to ./data/vista>"),
    )

check(
    "HF_HUB_OFFLINE=1",
    os.environ.get("HF_HUB_OFFLINE") == "1",
    os.environ.get("HF_HUB_OFFLINE", "<unset>"),
)

import torch  # noqa: E402

# is_available() only checks the device is visible — it stays True even when this
# PyTorch build has no compiled kernels for the GPU's compute capability (bit us
# twice on Vista: GH200/sm_90 silently unsupported by a Jetson-targeted build).
# Run a real op so a compute-capability mismatch actually fails this check.
try:
    _cuda_probe = (torch.randn(4, 4, device="cuda") @ torch.randn(4, 4, device="cuda")).sum().item()
    _cuda_ok = True
    _cuda_detail = f"{torch.cuda.get_device_name(0)}, matmul probe ok (sum={_cuda_probe:.2f})"
except Exception as exc:
    _cuda_ok = False
    _cuda_detail = f"{type(exc).__name__}: {exc}"

check("torch runs a real CUDA matmul (not just device visibility)", _cuda_ok, _cuda_detail)

nvidia_smi = shutil.which("nvidia-smi")
check("nvidia-smi on PATH", nvidia_smi is not None, nvidia_smi or "")

git_status = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_ROOT, capture_output=True, text=True
)
check(
    "git working tree clean", git_status.stdout.strip() == "", git_status.stdout.strip() or "clean"
)

git_head = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True
)
print(f"git HEAD: {git_head.stdout.strip()}")

## Checklist 2 — Profile, paths, and queue

In [ ]:
from training.profile import Profile  # noqa: E402
from training.queue import ExperimentQueue, ExperimentSpec, ExperimentStatus  # noqa: E402
from training.sweeps import QueuePopulator, default_grid  # noqa: E402

profile = Profile.load(PROFILE_NAME)
print(f"profile: {profile.name}  device={profile.device}  vlm={profile.vlm_model}")
for name, path in (
    ("data_root", profile.paths.data_root),
    ("ckpt_root", profile.paths.ckpt_root),
    ("mirror_root", profile.paths.mirror_root),
):
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name}: {path}")

check(
    "resolved paths are outside the repo (not silently defaulted)",
    all(
        str(p).startswith(("/scratch", "/work")) or "SCRATCH" in os.environ or "WORK" in os.environ
        for p in (profile.paths.data_root, profile.paths.ckpt_root, profile.paths.mirror_root)
    ),
    str(profile.paths.ckpt_root),
)

_model_dir = (
    Path(os.environ.get("HF_HOME", ""))
    / "hub"
    / ("models--" + profile.vlm_model.replace("/", "--"))
)
check(
    f"VLM weights prefetched ({profile.vlm_model})",
    _model_dir.is_dir(),
    str(_model_dir)
    if _model_dir.is_dir()
    else f"{_model_dir} missing — run scripts/prefetch_models.sh on a login node "
    "(lora/eval-vlm arms will fail without it)",
)

# mirror_root is the purge-safe destination (stands in for $WORK) — put session
# bookkeeping there so it survives $SCRATCH purges between allocation windows.
SESSION_STATE_PATH = profile.paths.mirror_root / "vista_session_state.json"
SESSION_LOG_PATH = profile.paths.mirror_root / "vista_session_log.md"
GPU_LOG_DIR = profile.paths.mirror_root / "gpu_logs"
GPU_LOG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
queue_path = REPO_ROOT / QUEUE_PATH
queue_path.parent.mkdir(parents=True, exist_ok=True)

# Queue statuses travel with the git repo, but corpus shards and checkpoints do not:
# a queue whose datagen is "done" on a machine with no corpus is another machine's
# history (e.g. the local pilot). Archive it and rebuild so this allocation actually
# generates its own data instead of instantly reporting "nothing to do".
if queue_path.exists():
    _snap = ExperimentQueue(queue_path).snapshot()
    _datagen_done = any(s.kind == "datagen" and s.status is ExperimentStatus.DONE for s in _snap)
    _has_shards = any(profile.paths.data_root.rglob("*.tar"))
    if _datagen_done and not _has_shards:
        _ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        _archive = profile.paths.mirror_root / f"queue-imported-{_ts}.yaml"
        shutil.move(str(queue_path), str(_archive))
        print(
            f"queue said datagen was done but {profile.paths.data_root} has no shards — "
            f"archived it to {_archive} and rebuilding fresh"
        )

if not queue_path.exists():
    print("bootstrapping queue: base entries + the H1xH3xH5 grid")
    base = [
        ExperimentSpec(name="datagen-mini", kind="datagen"),
        ExperimentSpec(name="pointnet-modelnet40", kind="pointnet", config={"epochs": 120}),
        ExperimentSpec(
            name="eval-h2-lifting",
            kind="eval",
            config={"detector": "gt", "lifting": "both", "split": "heldout"},
            requires=("datagen-mini",),
        ),
        ExperimentSpec(
            name="eval-sunrgbd-mini",
            kind="eval",
            config={"detector": "gt", "lifting": "both", "source": "sunrgbd", "max_samples": 50},
        ),
    ]
    QueuePopulator(queue_path).merge(base)

# Idempotent by name: safe every session; guarantees the hypothesis arms exist.
_added = QueuePopulator(queue_path).merge(default_grid().specs())
if _added:
    print(f"merged {len(_added)} hypothesis-grid arms into the queue")

queue = ExperimentQueue(queue_path)
snapshot_before = queue.snapshot()

print(f"{'name':<34}{'kind':<10}{'status':<10}{'retries':<8}requires")
for spec in snapshot_before:
    print(
        f"{spec.name:<34}{spec.kind:<10}{spec.status.value:<10}{spec.retries:<8}{','.join(spec.requires)}"
    )

pending_or_running = [
    s for s in snapshot_before if s.status in (ExperimentStatus.PENDING, ExperimentStatus.RUNNING)
]
check(
    "queue has runnable work",
    bool(pending_or_running) or EVOLVE,
    f"{len(pending_or_running)} pending/running of {len(snapshot_before)}",
)

## Checklist 3 — Canary preflight (WS0 guardrail)

Cheap, fast checks that must pass before this session spends any Vista GPU-hours. A prior pilot run drained an unattended queue at good utilization while silently producing scientifically void rows (3D IoU identically 0.000) — the canary is the guardrail against repeating that on Vista time.

In [ ]:
from training.canary import CanarySuite, SystemHalt  # noqa: E402

canary_ok = True
canary_detail = ""
try:
    CanarySuite.default(min_3d_iou=MIN_CANARY_IOU).assert_healthy()
except SystemHalt as exc:
    canary_ok = False
    canary_detail = str(exc)

check("canary preflight", canary_ok, canary_detail)
if not canary_ok:
    raise SystemHalt(
        "Canary preflight failed — refusing to launch the runner. Fix the underlying "
        "regression (see the canary_detail above and training/canary.py) before spending "
        "any more of the Vista window."
    )

## Checklist 4 — Launch this session's runner slice

In [ ]:
session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
gpu_log_path = GPU_LOG_DIR / f"gpu.{session_id}.csv"
runner_log_path = GPU_LOG_DIR / f"runner.{session_id}.log"

with open(gpu_log_path, "w") as gpu_log_fh:
    gpu_sampler = subprocess.Popen(
        [
            "nvidia-smi",
            "--query-gpu=timestamp,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader",
            "-l",
            "60",
        ],
        stdout=gpu_log_fh,
        stderr=subprocess.DEVNULL,
    )
    # Popen already dup'd the fd for the child; the `with` block closing our copy on exit is safe.

slice_minutes = SESSION_MINUTES - SAFETY_BUFFER_MINUTES
if slice_minutes <= 0:
    raise ValueError(
        f"SESSION_MINUTES ({SESSION_MINUTES}) must exceed SAFETY_BUFFER_MINUTES ({SAFETY_BUFFER_MINUTES})"
    )

cmd = [
    "timeout",
    "--signal=SIGTERM",
    f"--kill-after={KILL_AFTER_SECONDS}",
    f"{slice_minutes}m",
    sys.executable,
    "-m",
    "training.runner",
    "--profile",
    PROFILE_NAME,
    "--queue",
    QUEUE_PATH,
]
if EVOLVE:
    cmd.append("--evolve")

print(
    f"session {session_id}: launching for up to {slice_minutes} min "
    f"(SIGTERM then {KILL_AFTER_SECONDS}s grace)"
)
print(" ".join(cmd))

session_start = time.monotonic()
with open(runner_log_path, "w") as log_fh:
    proc = subprocess.Popen(
        cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
        log_fh.write(line)
    exit_code = proc.wait()
session_elapsed_min = (time.monotonic() - session_start) / 60.0

gpu_sampler.terminate()
try:
    gpu_sampler.wait(timeout=5)
except subprocess.TimeoutExpired:
    gpu_sampler.kill()

EXIT_MEANINGS = {
    0: "clean stop (drained or SIGTERM-checkpointed) ",
    3: "queue fully drained — nothing left to run",
    4: "canary preflight failed inside the runner (only reachable with EVOLVE=True)",
}
print(
    f"\nrunner exited {exit_code}: {EXIT_MEANINGS.get(exit_code, 'unexpected exit — check the log above')}"
)
print(f"elapsed: {session_elapsed_min:.1f} min")
check("runner exit code is a known-good code (0 or 3)", exit_code in (0, 3), f"exit={exit_code}")

## Checklist 5 — Verify the session actually made progress

A process that ran is not the same as progress: confirm the queue moved *and* that any experiment this session touched left behind a checkpoint newer than the session start, not just a claim that got orphaned.

In [ ]:
snapshot_after = queue.snapshot()
before_by_name = {s.name: s for s in snapshot_before}
changed = [
    (s.name, before_by_name[s.name].status.value, s.status.value)
    for s in snapshot_after
    if s.name in before_by_name and s.status != before_by_name[s.name].status
]

print(f"{'name':<34}{'before':<10}{'after':<10}")
for name, before, after in changed:
    print(f"{name:<34}{before:<10}{after:<10}")

check("queue status changed for at least one entry", bool(changed), f"{len(changed)} changed")

newly_done = [n for n, before, after in changed if after == "done"]
print(f"newly done this session: {newly_done or 'none'}")

In [ ]:
from training.checkpoint import CheckpointManager  # noqa: E402

touched_names = {n for n, _, _ in changed} | {
    s.name for s in snapshot_after if s.status is ExperimentStatus.RUNNING
}
session_start_wall = datetime.now(timezone.utc).timestamp() - session_elapsed_min * 60

stale_checkpoints = []
for name in sorted(touched_names):
    manager = CheckpointManager(profile.paths.ckpt_root / name)
    latest = manager.latest()
    if latest is None:
        stale_checkpoints.append((name, "no checkpoint directory at all"))
        continue
    mtime = latest.stat().st_mtime
    if mtime < session_start_wall - 60:  # 60s slack for clock skew
        stale_checkpoints.append((name, f"latest checkpoint predates this session ({latest.name})"))

check(
    "every touched experiment left a fresh checkpoint",
    not stale_checkpoints,
    "; ".join(f"{n}: {reason}" for n, reason in stale_checkpoints) or "all fresh",
)

## Checklist 6 — GPU utilization sanity

In [ ]:
import csv

util_samples = []
if gpu_log_path.exists():
    with open(gpu_log_path) as fh:
        for row in csv.reader(fh):
            if len(row) >= 2:
                try:
                    util_samples.append(float(row[1].strip().rstrip(" %")))
                except ValueError:
                    continue

if util_samples:
    avg_util = sum(util_samples) / len(util_samples)
    max_util = max(util_samples)
    print(f"GPU util over {len(util_samples)} samples: avg={avg_util:.1f}%  max={max_util:.1f}%")
    check(
        f"avg GPU util >= {LOW_GPU_UTIL_WARN_PCT:.0f}%",
        avg_util >= LOW_GPU_UTIL_WARN_PCT,
        f"avg={avg_util:.1f}%",
    )
else:
    print("no GPU samples captured (session shorter than the 60s sampling interval?)")
    check("GPU utilization sampled", False, "no samples")

## Checklist 7 — Session handoff log

Appends a record so the *next* session — possibly launched hours later with no shared chat context — knows exactly what state the queue and window budget are in.

In [ ]:
state = {"window_started_utc": None, "window_hours": WINDOW_HOURS, "sessions": []}
if SESSION_STATE_PATH.exists():
    state = json.loads(SESSION_STATE_PATH.read_text())

if state["window_started_utc"] is None:
    state["window_started_utc"] = datetime.now(timezone.utc).isoformat()

session_record = {
    "session_id": session_id,
    "started_utc": datetime.fromtimestamp(session_start_wall, tz=timezone.utc).isoformat(),
    "elapsed_minutes": round(session_elapsed_min, 1),
    "exit_code": exit_code,
    "queue_changed": changed,
    "newly_done": newly_done,
    "stale_checkpoints": stale_checkpoints,
    "gpu_avg_util_pct": round(avg_util, 1) if util_samples else None,
    "checklist": [{"label": label, "ok": ok, "detail": detail} for label, ok, detail in checklist],
}
state["sessions"].append(session_record)
SESSION_STATE_PATH.write_text(json.dumps(state, indent=2))

window_started = datetime.fromisoformat(state["window_started_utc"])
window_elapsed_h = (datetime.now(timezone.utc) - window_started).total_seconds() / 3600.0
window_remaining_h = WINDOW_HOURS - window_elapsed_h

with open(SESSION_LOG_PATH, "a") as fh:
    fh.write(f"## Session {session_id}\n")
    fh.write(f"- elapsed: {session_elapsed_min:.1f} min, exit={exit_code}\n")
    fh.write(f"- queue changes: {changed or 'none'}\n")
    fh.write(f"- newly done: {newly_done or 'none'}\n")
    fh.write(f"- stale checkpoints: {stale_checkpoints or 'none'}\n")
    fh.write(f"- gpu avg util: {round(avg_util, 1) if util_samples else 'n/a'}%\n")
    fh.write(f"- window remaining: {window_remaining_h:.1f}h of {WINDOW_HOURS}h\n\n")

print(f"session log: {SESSION_LOG_PATH}")
print(
    f"window remaining: {window_remaining_h:.1f}h of {WINDOW_HOURS}h ({len(state['sessions'])} sessions run)"
)

## Final checklist summary

In [ ]:
print("=" * 60)
print(f"SESSION {session_id} SUMMARY")
print("=" * 60)
for label, ok, detail in checklist:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f" — {detail}" if detail else ""))

all_ok = all(ok for _, ok, _ in checklist)
queue_drained = exit_code == 3

print()
if not all_ok:
    print(
        "ACTION: something above FAILed — investigate before the next session; "
        "do not assume unattended progress happened."
    )
elif queue_drained:
    print(
        "ACTION: queue is fully drained. Nothing left to run — stop here, or set "
        "EVOLVE=True / repopulate experiments/queue.yaml (training.sweeps) for more work."
    )
elif window_remaining_h <= 0:
    print("ACTION: allocation window is used up. Stop; resume in the next window.")
else:
    print(
        f"ACTION: all checks passed, {window_remaining_h:.1f}h remain in the window — "
        "re-run this notebook top-to-bottom for the next session."
    )